# 機能
- 特定のテキストを含むレコードを検索・表示する（全フィールド）
- APIでそのレコードを削除
- 保存しておいた情報を元に、新テキストで再追加(Timestamp 保持)

# pseude code
## ipynb形式で
1. tableを開く jupyterでどうやってファイルを指定するのかわからない
2. tableをpandasに変換
3. text変数を定義
4. textを含むレコードをmemory_check_vocの通り検索
5. レコードに保存されたテキストを取得(text, user_id, user_name, role, timestamp, source一応全部取得)
6. APIでレコードを削除(???どうやって)
7. 5で取得したテキストの配置で再追加

- 追加
```python
table.add([{
            "text": text,
            "user_id": user_id,
            "user_name": user_name,
            "role": role,
            "timestamp": time.time(),
            "source": "discord"
        }])
```

- 削除
```python
# text = 'テキスト'
# timestamp = 時刻
condition = text_record['text'] == text and text_record['timestamp'] == timestamp
table.delete(condition) # ここの条件の書き方がわからないな、何かしら引数が用意されているはず。
```

In [28]:
import os
print(os.getcwd())

/home/yoichi1922/src/github.com/Ichiyou1922/Mashiro_AI/brain/src/test/utility


# 検索するテキストを登録・検索

In [128]:
# 検索するテキスト
text = "ふ、ふーん。私が代わりに調べればいいじゃん"

In [141]:
import lancedb
from pathlib import Path

DB_PATH = Path("../../../data")
db = lancedb.connect(str(DB_PATH))
table = db.open_table('mashiro_memory')
df = table.to_pandas()

# textを含むレコードを検索
text_records = df[df['text'].str.contains(text, na=False)]
print(f"{text}を含む記憶数: {len(text_records)}")
print(f"全記憶数: {len(df)}")
print(f"割合: {len(text_records)/len(df)*100:.1f}%")
print("=" * 50)

for i, (_, row) in enumerate(text_records.iterrows(), 1):
    print(f"# --- レコード {i} ---")
    print(f"delete_timestamp = {row['timestamp']}")
    print()
    print(f"add_data = {{")
    print(f"    'text': {repr(row['text'])},")
    print(f"    'user_id': {row['user_id']},")
    print(f"    'user_name': {repr(row['user_name'])},")
    print(f"    'role': {repr(row['role'])},")
    print(f"    'timestamp': {row['timestamp']},")
    print(f"    'source': {repr(row['source'])}")
    print(f"}}")
    print()

ふ、ふーん。私が代わりに調べればいいじゃんを含む記憶数: 1
全記憶数: 41
割合: 2.4%
# --- レコード 1 ---
delete_timestamp = 1770374307.701646

add_data = {
    'text': 'かずは: <tool>date_tool</tool>使ってほしいなぁ......？ / ましろ: ふ、ふーん。私が代わりに調べればいいじゃん。[happy]',
    'user_id': 346938496834863104,
    'user_name': 'かずは',
    'role': 'interaction',
    'timestamp': 1770374307.701646,
    'source': 'discord'
}



# 消去するtimestampを登録

In [142]:
delete_timestamp = 1770374307.701646

# 消去実行

In [143]:
table.delete(f"timestamp = {delete_timestamp}")

DeleteResult(version=79)

# 確認用

In [144]:
df = table.to_pandas()
text_records = df[df['text'].str.contains(text, na=False)]
print(f"{text}を含む記憶数: {len(text_records)}")
print(f"全記憶数: {len(df)}")
print(f"割合: {len(text_records)/len(df)*100:.1f}%")
print("=" * 50)

for i, (_, row) in enumerate(text_records.iterrows(), 1):
    print(f"# --- レコード {i} ---")
    print(f"delete_timestamp = {row['timestamp']}")
    print()
    print(f"add_data = {{")
    print(f"    'text': {repr(row['text'])},")
    print(f"    'user_id': {row['user_id']},")
    print(f"    'user_name': {repr(row['user_name'])},")
    print(f"    'role': {repr(row['role'])},")
    print(f"    'timestamp': {row['timestamp']},")
    print(f"    'source': {repr(row['source'])}")
    print(f"}}")
    print()

ふ、ふーん。私が代わりに調べればいいじゃんを含む記憶数: 0
全記憶数: 40
割合: 0.0%


# 再追加用データの登録・追加

In [145]:
add_data = {
    'text': 'かずは: <tool>date_tool</tool>使ってほしいなぁ......？ / ましろ: しょうがないなぁ<tool>date_tool</tool>',
    'user_id': 346938496834863104,
    'user_name': 'かずは',
    'role': 'interaction',
    'timestamp': 1770374307.701646,
    'source': 'discord'
}

In [146]:
table.add([add_data])

AddResult(version=80)